# Clase 7 — ¿Cómo aprende a hablar una máquina?

> *“Al final, un LLM no es más que un sistema que aprendió, carácter por carácter,
> qué viene después de qué.”*

En esta clase entendés la **intuición** detrás de los modelos de lenguaje:
cómo tokenizar texto, cómo el modelo “ve” los datos y cuál es la arquitectura
Transformer a grandes rasgos.

Al final tenés una **demo interactiva** para jugar con tokenización y generación.

### Lo que vas a llevarte

| Sección | Concepto |
|---|---|
| 1 | Instalación y verificación del entorno |
| 2 | ¿Qué es un modelo de lenguaje? Intuición |
| 3 | El dataset: qué texto va a aprender el modelo |
| 4 | Tokenización: de texto a números |
| 5 | Ventana deslizante: cómo armar el dataset |
| 6 | Demo interactiva: jugá con el tokenizador |
| 7 | Arquitectura Transformer: vista conceptual |
| 8 | Ejercicio conceptual |

> **Prerequisito:** Python 3.9+ con PyTorch instalado (ver sección 1).
> La clase 8 construye el modelo completo; aquí solo trabajamos con datos y conceptos.


---
## 1. Instalación de dependencias

Solo necesitamos **PyTorch** y **Matplotlib**. No hay más deps.

---

### 🍎 Mac con Apple Silicon (M1 / M2 / M3 / M4)

PyTorch soporta natively Metal Performance Shaders (MPS) en Apple Silicon:

```bash
pip install torch torchvision torchaudio matplotlib
```

> La aceleración MPS se activa automáticamente con `device = "mps"` — lo verás más adelante.

---

### 🍎 Mac con Intel

```bash
pip install torch torchvision torchaudio matplotlib
```

> No hay aceleración de GPU disponible en Mac Intel. Usará CPU.

---

### 🐧 Linux (CPU)

```bash
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
pip install matplotlib
```

### 🐧 Linux (GPU NVIDIA — CUDA 12.1)

```bash
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
pip install matplotlib
```

---

### 🪟 Windows (CPU)

```powershell
pip install torch torchvision torchaudio matplotlib
```

### 🪟 Windows (GPU NVIDIA — CUDA 12.1)

```powershell
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
pip install matplotlib
```

### 🪟 Windows con Conda (recomendado en Windows)

```powershell
conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia
conda install matplotlib
```

---

> 📌 Verificá siempre la versión correcta de CUDA para tu GPU en [pytorch.org/get-started](https://pytorch.org/get-started/locally/)

In [1]:
"""
import subprocess, sys

# Instala las dependencias mínimas requeridas para este notebook
subprocess.run(
    [sys.executable, "-m", "pip", "install", "torch", "matplotlib", "--quiet"],
    check=True
)
print("✅ Instalación completada")
"""

'\nimport subprocess, sys\n\n# Instala las dependencias mínimas requeridas para este notebook\nsubprocess.run(\n    [sys.executable, "-m", "pip", "install", "torch", "matplotlib", "--quiet"],\n    check=True\n)\nprint("✅ Instalación completada")\n'

---
### Verificación del entorno y detección de dispositivo


Primero verificamos que PyTorch esté instalado y detectamos qué acelerador de hardware tenemos disponible:

| Dispositivo | Significado |
|---|---|
| `cpu` | Solo procesador — funciona siempre, más lento |
| `mps` | Apple Silicon GPU via Metal — Mac M1/M2/M3/M4 |
| `cuda` | GPU NVIDIA — el más rápido para deep learning |

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math, time, random

print(f"✅ PyTorch versión : {torch.__version__}")
print()

# Detectar el mejor dispositivo disponible
if torch.cuda.is_available():
    device = torch.device("cuda")
    nombre_gpu = torch.cuda.get_device_name(0)
    memoria_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🟢 NVIDIA GPU disponible: {nombre_gpu} ({memoria_gb:.1f} GB VRAM)")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🟢 Apple Silicon GPU (Metal MPS) disponible")
else:
    device = torch.device("cpu")
    print("🔵 Usando CPU (sin GPU disponible)")

print(f"\n📍 Dispositivo seleccionado: {device}")
print("   Este notebook corre bien en CPU — el modelo es pequeño a propósito.")

✅ PyTorch versión : 2.11.0

🟢 Apple Silicon GPU (Metal MPS) disponible

📍 Dispositivo seleccionado: mps
   Este notebook corre bien en CPU — el modelo es pequeño a propósito.


---
## 2. ¿Qué es un modelo de lenguaje? Intuición

Un **modelo de lenguaje** es un sistema que asigna probabilidades a secuencias de texto.
En la práctica, aprende a responder a la pregunta:

> *“Dado este contexto, ¿cuál es el siguiente token más probable?”*

### Del n-grama al Transformer

| Generación | Idea | Limitación |
|---|---|---|
| **N-gramas** (1990s) | Contar frecuencias de N palabras seguidas | Solo contexto corto, tabla enorme |
| **RNN/LSTM** (2000s-2010s) | Estado oculto que “recuerda” el pasado | Gradientes que desaparecen, lento |
| **Transformer** (2017–) | Atención: cada token mira a todos los anteriores | Paralelo, escalable, base de GPT/LLaMA |

### El ciclo de entrenamiento en una línea

```
texto de entrenamiento  →  tokenizar  →  predecir siguiente token  →  medir error  →  ajustar pesos  ↻
```

Repetido miles de millones de veces, el modelo aprende gramática, hechos, lógica y hasta estilo.

> **Clave:** la tarea de entrenamiento es simple (predecir el siguiente token),
> pero escalarla genera comportamientos emergentes sorprendentes.


---
## 3. El dataset: ¿qué texto va a aprender el modelo?

Para que el entrenamiento sea rápido y comprensible, usaremos un texto corto directamente en el código.  
El modelo aprenderá a **reproducir y continuar** el estilo de ese texto.

### ¿Por qué un texto corto?

- Un texto de ~2000 caracteres tarda segundos en entrenar (no horas)
- Podés ver el resultado inmediatamente
- Los conceptos son 100% iguales que con datasets de millones de tokens

### ¿Qué texto usamos?

Usamos un texto sobre inteligencia artificial. El modelo aprenderá a generar texto similar.  
Podés reemplazar `TEXTO` con cualquier contenido que quieras (un libro, código, letras de canciones).

> 💡 **Regla práctica:** Para que un modelo character-level aprenda algo útil,  
> necesitás al menos 100x más caracteres que parámetros del modelo.

In [3]:
# El texto que el modelo aprenderá a imitar
# Puedes reemplazarlo con cualquier texto (más texto = mejor resultado)
TEXTO = """
La inteligencia artificial es una rama de la informática que busca crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana. Entre estas tareas se incluyen el reconocimiento de voz, la comprensión del lenguaje natural, el aprendizaje y la resolución de problemas.

El aprendizaje automático es un subconjunto de la inteligencia artificial que permite a los sistemas aprender y mejorar a partir de la experiencia sin ser programados explícitamente. El aprendizaje automático se centra en el desarrollo de programas que pueden acceder a datos y utilizarlos para aprender por sí mismos.

El aprendizaje profundo es un método de aprendizaje automático que enseña a las computadoras a hacer lo que naturalmente hace el ser humano: aprender con el ejemplo. El aprendizaje profundo es una técnica clave detrás de los autos sin conductor, que permite al automóvil reconocer una señal de pare o distinguir a un peatón de una farola.

Las redes neuronales artificiales están inspiradas en el cerebro humano. Estas redes aprenden de grandes cantidades de datos. Al igual que los humanos aprenden de la experiencia, el algoritmo de la red neuronal aprende a realizar tareas de clasificación directamente de imágenes, texto o sonido.

Los transformers revolucionaron el procesamiento del lenguaje natural. Su mecanismo de atención permite al modelo enfocarse en partes relevantes de la entrada al generar cada parte de la salida. Esta arquitectura es la base de modelos como GPT, BERT y LLaMA.

Entrenar un modelo de lenguaje consiste en presentarle millones de ejemplos de texto y ajustar sus parámetros para que sea capaz de predecir el siguiente token. Con suficientes datos y capacidad, el modelo aprende gramática, hechos, razonamiento y hasta creatividad.
"""

print(f"📄 Texto cargado:")
print(f"   Total de caracteres : {len(TEXTO):,}")
print(f"   Total de palabras   : {len(TEXTO.split()):,}")
print(f"\nPrimeros 200 caracteres:")
print(f"{'─'*50}")
print(TEXTO[:200])

📄 Texto cargado:
   Total de caracteres : 1,781
   Total de palabras   : 275

Primeros 200 caracteres:
──────────────────────────────────────────────────

La inteligencia artificial es una rama de la informática que busca crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana. Entre estas tareas se incluyen el reconocim


---
## 4. Tokenización: de texto a números

Los modelos no entienden caracteres ni palabras — solo **números**.  
La **tokenización** es el proceso de convertir texto en una secuencia de enteros.

### Tipos de tokenización

| Tipo | Unidad | Vocabulario | Ejemplo |
|---|---|---|---|
| **Character-level** | 1 carácter | ~100 tokens | `h`, `o`, `l`, `a` |
| **Word-level** | 1 palabra | ~50,000 tokens | `hola`, `mundo` |
| **BPE (subword)** | fragmentos | ~32,000-100,000 | `hol`, `##a` |
| **SentencePiece** | subpalabras | variable | `▁hola`, `▁mundo` |

GPT-4 usa BPE con ~100,000 tokens. Nosotros usaremos **character-level** por ser el más simple de entender.

### El vocabulario

Es el conjunto de todos los tokens únicos que puede manejar el modelo.  
Con character-level tokenization, el vocabulario son **todos los caracteres únicos** del texto.

```
texto  →  "hola"  →  encode  →  [7, 14, 11, 0]  →  el modelo
el modelo  →  [7, 14, 11, 0]  →  decode  →  "hola"
```

In [4]:
# ─── Construir el vocabulario ─────────────────────────────────────────────────

# El vocabulario son todos los caracteres únicos del texto, ordenados
vocabulario = sorted(set(TEXTO))
vocab_size   = len(vocabulario)

print(f"📚 Vocabulario construido:")
print(f"   Tamaño: {vocab_size} caracteres únicos")
print(f"   Caracteres: {''.join(vocabulario[:46])} ")
print()

📚 Vocabulario construido:
   Tamaño: 46 caracteres únicos
   Caracteres: 
 ,.:ABCEGLMPRSTabcdefghijklmnopqrstuvxyzáéíñó 



In [5]:
# ─── Crear las funciones de encode y decode ───────────────────────────────────

# char → int  (cada carácter mapea a un índice)
char_a_int = {char: idx for idx, char in enumerate(vocabulario)}

# int → char  (cada índice mapea a un carácter)
int_a_char = {idx: char for idx, char in enumerate(vocabulario)}

def encode(texto: str) -> list[int]:
    """Convierte un string en una lista de enteros."""
    return [char_a_int[c] for c in texto]

def decode(tokens: list[int]) -> str:
    """Convierte una lista de enteros en un string."""
    return "".join(int_a_char[i] for i in tokens)

In [6]:
# ─── Demostración ─────────────────────────────────────────────────────────────
texto_ejemplo  = "hola mundo"
tokens_ejemplo = encode(texto_ejemplo)
texto_recup    = decode(tokens_ejemplo)

print(f"🔤 Ejemplo de tokenización:")
print(f"   Original  : '{texto_ejemplo}'")
print(f"   Tokens    : {tokens_ejemplo}")
print(f"   Recuperado: '{texto_recup}'")
print(f"   ¿Igual?   : {texto_ejemplo == texto_recup}")

# ─── Codificar el texto completo ──────────────────────────────────────────────
datos = torch.tensor(encode(TEXTO), dtype=torch.long)

print(f"\n📊 Dataset codificado:")
print(f"   Shape  : {datos.shape}")
print(f"   Dtype  : {datos.dtype}")
print(f"   Primeros 20 tokens: {datos[:20].tolist()}")

🔤 Ejemplo de tokenización:
   Original  : 'hola mundo'
   Tokens    : [23, 30, 27, 16, 1, 28, 36, 29, 19, 30]
   Recuperado: 'hola mundo'
   ¿Igual?   : True

📊 Dataset codificado:
   Shape  : torch.Size([1781])
   Dtype  : torch.int64
   Primeros 20 tokens: [0, 10, 16, 1, 24, 29, 35, 20, 27, 24, 22, 20, 29, 18, 24, 16, 1, 16, 33, 35]


---
## 5. Preparar los datos: ventana deslizante

El modelo aprende a **predecir el siguiente token** dado un contexto.  
Para esto, dividimos el texto en pares **(entrada X, objetivo Y)** usando una **ventana deslizante**.

### ¿Cómo funciona?

Supongamos que el texto codificado es: `[5, 3, 8, 1, 6, 2]` y el tamaño de contexto es 4.

```
X (entrada)         →   Y (lo que debe predecir)
─────────────────────────────────────────────────
[5]                 →   3
[5, 3]              →   8
[5, 3, 8]           →   1
[5, 3, 8, 1]        →   6     ← ventana completa
[3, 8, 1, 6]        →   2     ← la ventana desliza
```

En la práctica, esto se implementa extrayendo bloques de longitud `contexto + 1`:  
`X = bloque[:-1]` (todos menos el último) y `Y = bloque[1:]` (todos menos el primero).

### División train/validation

Separamos el 20% del texto como **conjunto de validación** para medir si el modelo está aprendiendo  
o memorizando (overfitting).

In [7]:
# ─── Hiperparámetros del dataset ──────────────────────────────────────────────
CONTEXTO   = 64   # cuántos tokens mira el modelo como "contexto" (tamaño de la ventana)
BATCH_SIZE = 32   # cuántos ejemplos se procesan en paralelo por paso
SPLIT      = 0.8  # 80% para entrenamiento, 20% para validación

# ─── División train / validación ─────────────────────────────────────────────
n_train = int(len(datos) * SPLIT)
datos_train = datos[:n_train]
datos_val   = datos[n_train:]

print(f"📦 División del dataset:")
print(f"   Total de tokens        : {len(datos):,}")
print(f"   Tokens de entrenamiento: {len(datos_train):,} ({SPLIT*100:.0f}%)")
print(f"   Tokens de validación   : {len(datos_val):,} ({(1-SPLIT)*100:.0f}%)")
print()

# ─── Función para obtener batches aleatorios ──────────────────────────────────
def get_batch(split: str):
    """
    Genera un mini-batch de pares (X, Y) aleatorios del dataset.
    
    Parameters:
        split: 'train' o 'val'
    
    Returns:
        x: tensor de shape (BATCH_SIZE, CONTEXTO) — las entradas
        y: tensor de shape (BATCH_SIZE, CONTEXTO) — los objetivos (x desplazado 1)
    """
    data = datos_train if split == 'train' else datos_val
    
    # Elegir BATCH_SIZE posiciones de inicio aleatorias
    # (cada posición debe tener suficiente texto por delante)
    indices = torch.randint(len(data) - CONTEXTO, (BATCH_SIZE,))

    # Para cada índice, extraer una ventana de CONTEXTO tokens
    x = torch.stack([data[i : i + CONTEXTO]     for i in indices])
    y = torch.stack([data[i + 1 : i + CONTEXTO + 1] for i in indices])  # desplazado 1
    
    return x.to(device), y.to(device)

# ─── Demostración de un batch ────────────────────────────────────────────────
x_demo, y_demo = get_batch('train')

print(f"🎲 Ejemplo de batch:")
print(f"   X shape: {x_demo.shape}  — ({BATCH_SIZE} ejemplos, {CONTEXTO} tokens cada uno)")
print(f"   Y shape: {y_demo.shape}  — (el objetivo: X desplazado 1)")
print()
print(f"Primer ejemplo del batch:")
print(f"   X[0] = {x_demo[0, :10].tolist()} ... (primeros 10 tokens)")
print(f"   Y[0] = {y_demo[0, :10].tolist()} ... (los mismos, desplazados 1)")
print()
print(f"Primer ejemplo decodificado:")
print(f"   X[0] → '{decode(x_demo[0].tolist())[:50]}'")

📦 División del dataset:
   Total de tokens        : 1,781
   Tokens de entrenamiento: 1,424 (80%)
   Tokens de validación   : 357 (20%)

🎲 Ejemplo de batch:
   X shape: torch.Size([32, 64])  — (32 ejemplos, 64 tokens cada uno)
   Y shape: torch.Size([32, 64])  — (el objetivo: X desplazado 1)

Primer ejemplo del batch:
   X[0] = [1, 24, 29, 35, 20, 27, 24, 22, 20, 29] ... (primeros 10 tokens)
   Y[0] = [24, 29, 35, 20, 27, 24, 22, 20, 29, 18] ... (los mismos, desplazados 1)

Primer ejemplo decodificado:
   X[0] → ' inteligencia artificial que permite a los sistema'


---
## 6. Demo interactiva: jugá con el tokenizador

Ahora que tenés las funciones `encode` y `decode`, usálas para explorar cómo
el modelo “ve” cualquier texto. Probá con palabras en español, emojis, código...


In [8]:
# ── Demo interactiva del tokenizador ────────────────────────────────────────
# Cam biá cualquiera de estos textos y re-ejecutá la celda

textos_prueba = [
    "hola mundo",
    "inteligencia artificial",
    "GPT aprende del contexto",
    "el modelo predice el siguiente token",
]

print("Tokenizador character-level")
print("=" * 50)
for t in textos_prueba:
    # Filtrar caracteres que no estén en el vocabulario
    t_filtrado = "".join(c for c in t if c in char_a_int)
    tokens = encode(t_filtrado)
    recup  = decode(tokens)
    print(f"  Texto    : {t_filtrado!r}")
    print(f"  Tokens   : {tokens}")
    print(f"  N tokens : {len(tokens)}")
    print(f"  Decoded  : {recup!r}")
    print()

print(f"Vocab size: {vocab_size} caracteres únicos")
print(f"Vocabulario: {''.join(vocabulario)}")

# ¿Cuántos caracteres únicos tiene el texto de entrenamiento?
print(f"\nComparación con BPE (GPT-4):")
print(f"  Nuestro vocab: {vocab_size} tokens")
print(f"  GPT-4 vocab  : ~100,000 tokens  (subpalabras, no caracteres)")
print(f"  Consecuencia : GPT-4 usa ~{100000//vocab_size}x menos tokens para el mismo texto")


Tokenizador character-level
  Texto    : 'hola mundo'
  Tokens   : [23, 30, 27, 16, 1, 28, 36, 29, 19, 30]
  N tokens : 10
  Decoded  : 'hola mundo'

  Texto    : 'inteligencia artificial'
  Tokens   : [24, 29, 35, 20, 27, 24, 22, 20, 29, 18, 24, 16, 1, 16, 33, 35, 24, 21, 24, 18, 24, 16, 27]
  N tokens : 23
  Decoded  : 'inteligencia artificial'

  Texto    : 'GPT aprende del contexto'
  Tokens   : [9, 12, 15, 1, 16, 31, 33, 20, 29, 19, 20, 1, 19, 20, 27, 1, 18, 30, 29, 35, 20, 38, 35, 30]
  N tokens : 24
  Decoded  : 'GPT aprende del contexto'

  Texto    : 'el modelo predice el siguiente token'
  Tokens   : [20, 27, 1, 28, 30, 19, 20, 27, 30, 1, 31, 33, 20, 19, 24, 18, 20, 1, 20, 27, 1, 34, 24, 22, 36, 24, 20, 29, 35, 20, 1, 35, 30, 26, 20, 29]
  N tokens : 36
  Decoded  : 'el modelo predice el siguiente token'

Vocab size: 46 caracteres únicos
Vocabulario: 
 ,.:ABCEGLMPRSTabcdefghijklmnopqrstuvxyzáéíñó

Comparación con BPE (GPT-4):
  Nuestro vocab: 46 tokens
  GPT-4 vocab  : ~100,0

---
## 7. Arquitectura Transformer: vista conceptual

En la clase siguiente vas a implementar cada pieza. Aquí la visión de pájaro:

```
Token  →  Embedding  →  [Bloque Transformer] x N  →  Proyección  →  Probabilidades
```

### Componentes clave

| Componente | Qué hace |
|---|---|
| **Token embedding** | Convierte cada token (entero) en un vector de números reales |
| **Positional embedding** | Agrega información de posición (el Transformer no conoce el orden por defecto) |
| **Self-Attention** | Cada token mira todos los anteriores y pondera cuánto le importa cada uno |
| **Feed-Forward** | Red neuronal simple que procesa cada token de forma independiente |
| **LayerNorm** | Normaliza las activaciones para estabilizar el entrenamiento |
| **Conexión residual** | Suma la entrada con la salida de cada bloque (evita vanishing gradient) |

### ¿Por qué el mecanismo de atención es la clave?

La atención resuelve la **dependencia de largo alcance**: saber que en
*“El modelo que entrené ayer **falló**”* la palabra “falló” se refiere a “modelo”,
no a “enter” ni a “ayer”.

```
Atención(Q, K, V) = softmax(QKᵀ / √d_k) · V
```

- **Q (Query):** “¿qué información busco?”
- **K (Key):**   “¿qué información tengo?”
- **V (Value):** la información en sí

> En la clase 8 vas a ver este código real, celda por celda.


---
## 8. Ejercicio conceptual

### Parte A — Tokenización

1. ¿Cuántos tokens ocupa la palabra `“antidisestablishmentarianism”` con tokenización
   **character-level**? ¿Y si usás BPE (GPT-4)? ¿Cómo afecta eso al contexto?

2. Si tu modelo tiene un contexto de **64 tokens** y usás character-level,
   ¿cuántas palabras (aprox.) caben en ese contexto?
   ¿Y con BPE de 32K vocab?

3. ¿Por qué un modelo character-level puede generar palabras inventadas
   que un modelo BPE no generaría? ¿Es una ventaja o desventaja?

### Parte B — Ventana deslizante

Dado el texto: `"gato come fish"`

Con `CONTEXTO = 4` y tokenización character-level:
1. Listá los primeros **5 pares (X, Y)** que generaría la ventana deslizante.
2. ¿Cuántos pares (X, Y) totales puede generar ese texto?

### Parte C — Arquitectura (reflexión)

Sin ver código todavía, respondé con tus palabras:
- ¿Por qué el Transformer necesita **positional embeddings**
  si la atención ya “ve” todos los tokens?
- ¿Qué problema resuelve la **máscara causal** durante el entrenamiento?

> En la clase 8 vas a implementar cada una de estas piezas en código real.
